# nb27 - CNC cosmology posteriors (FLAMINGO arnaudB1 cluster counts)

Cluster-number-count inference from the FLAMINGO mock binned counts
`data/cnc/N2d_z_q_bin_arnaudB1.txt` (10 z x 5 q, B=1 / A10 selection) using
`cosmocnc_jax` (hmfast HMF), SZiFi y0 scaling relation.

**Sampled (5):** Omega_m ~ U(0.2,0.5), sigma8 ~ U(0.7,0.9), alpha(=alpha_szifi) ~
N(1.0,0.1), A_SZ ~ N(-4.3054,0.0414) (10% amplitude), sigma_lnY ~ N(0.173,0.023).
**Fixed:** B=1 (no mass bias); h, omega_b, n_s, tau, m_nu (D3A). Chains:
`chains/cnc_cosmo_arnaudB1/` (Y_5R500c, N=2392) and
`chains/cnc_cosmo_arnaudB1_Y500c/` (Y_R500c, N=2023).

In [1]:
import os
os.environ["XLA_FLAGS"] = "--xla_profile_fraction=0.10"

import numpy as np
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from getdist import loadMCSamples, plots

REPO="/scratch/scratch-lxu/flamingo_repo"
CHAIN=os.path.join(REPO,"chains","cnc_cosmo_arnaudB1","cnc_cosmo")
N_Y5R500C = 2392   # q>5 from Y_5R500c catalogue (B=1)
N_Y500C = 2023     # q>5 from Y_R500c catalogue (B=1)
OUTDIR=os.path.join(REPO,"figures","nb27_cnc_cosmo_contours"); os.makedirs(OUTDIR,exist_ok=True)

# FLAMINGO truth / fiducials
OM_TRUTH, S8_TRUTH = 0.306, 0.808
TRUTH={"Omega_m":0.306,"sigma8":S8_TRUTH,"S8":S8_TRUTH*np.sqrt(0.306/0.3),"h":0.681,"omega_b":0.022539,"n_s":0.967,"alpha_SZ":1.12,"A_SZ":-4.3054,"sigma_lnY":0.173}
PARS=["Omega_m","sigma8","S8","h","omega_b","n_s","alpha_SZ","A_SZ","sigma_lnY"]

s=loadMCSamples(CHAIN, settings={"ignore_rows":0.3})
p=s.getParams()
s.addDerived(p.sigma8*np.sqrt(p.Omega_m/0.3), name="S8", label=r"S_8\equiv\sigma_8(\Omega_m/0.3)^{0.5}")
print("CNC posterior B=1 Y_5R500c (mean +/- std):")
for nm in PARS:
    print(f"  {nm:10s} = {s.mean(nm):.4f} +/- {s.std(nm):.4f}")
print(f"\nFLAMINGO truth: Omega_m={OM_TRUTH}, sigma8={S8_TRUTH}, S8={S8_TRUTH*np.sqrt(OM_TRUTH/0.3):.3f}")

CHAIN_B1_Y500C = os.path.join(REPO, "chains", "cnc_cosmo_arnaudB1_Y500c", "cnc_cosmo")
if not os.path.exists(CHAIN_B1_Y500C + ".1.txt"):
    print("\nY_R500c chain not found:", CHAIN_B1_Y500C)
    print("run: bash cobaya/run_cnc_cosmo_B1_Y500c.sh")
    s1_Y500c = None
else:
    s1_Y500c = loadMCSamples(CHAIN_B1_Y500C, settings={"ignore_rows": 0.3})
    p1 = s1_Y500c.getParams()
    s1_Y500c.addDerived(p1.sigma8 * np.sqrt(p1.Omega_m / 0.3), name="S8",
                        label=r"S_8\equiv\sigma_8(\Omega_m/0.3)^{0.5}")
    print(f"\nCNC posterior B=1 Y_R500c (N={N_Y500C}; same priors as Y_5R500c chain):")
    for nm in PARS:
        if nm not in s1_Y500c.getParamNames().list():
            continue
        print(f"  {nm:10s} = {s1_Y500c.mean(nm):.4f} +/- {s1_Y500c.std(nm):.4f}")
    print("B=1 Y_5R500c S8 = %.4f +/- %.4f" % (s.mean("S8"), s.std("S8")))
    print("B=1 Y_R500c  S8 = %.4f +/- %.4f" % (s1_Y500c.mean("S8"), s1_Y500c.std("S8")))


CNC posterior B=1 Y_5R500c (mean +/- std):
  Omega_m    = 0.3658 +/- 0.0211
  sigma8     = 0.7162 +/- 0.0117
  S8         = 0.7904 +/- 0.0247
  h          = 0.6837 +/- 0.0236
  omega_b    = 0.0225 +/- 0.0020
  n_s        = 0.9660 +/- 0.0145
  alpha_SZ   = 1.1304 +/- 0.0491
  A_SZ       = -4.3098 +/- 0.0543
  sigma_lnY  = 0.1721 +/- 0.0235

FLAMINGO truth: Omega_m=0.306, sigma8=0.808, S8=0.816

CNC posterior B=1 Y_R500c (N=2023; same priors as Y_5R500c chain):
  Omega_m    = 0.2941 +/- 0.0115
  sigma8     = 0.7830 +/- 0.0162
  S8         = 0.7749 +/- 0.0135
  h          = 0.6746 +/- 0.0242
  omega_b    = 0.0227 +/- 0.0020
  n_s        = 0.9657 +/- 0.0135
  alpha_SZ   = 1.1005 +/- 0.0457
  sigma_lnY  = 0.1744 +/- 0.0222
B=1 Y_5R500c S8 = 0.7904 +/- 0.0247
B=1 Y_R500c  S8 = 0.7749 +/- 0.0135


## Triangle of all sampled parameters (B=1: $Y_{5R500c}$ vs $Y_{R500c}$)

In [2]:
g=plots.get_subplot_plotter(width_inch=11.0)
g.settings.alpha_filled_add=0.7; g.settings.title_limit_fontsize=10
chains = [s] if s1_Y500c is None else [s, s1_Y500c]
colors = ["#1f77b4"] if s1_Y500c is None else ["#1f77b4", "#2ca02c"]
labels = [f"B=1 Y_5R500c (N={N_Y5R500C})"] if s1_Y500c is None else [
    f"B=1 Y_5R500c (N={N_Y5R500C})", f"B=1 Y_R500c (N={N_Y500C})"]
g.triangle_plot(chains, PARS, filled=True, contour_colors=colors,
              legend_labels=labels, legend_loc="upper right",
              title_limit=1, markers=TRUTH)
g.fig.suptitle(r"CNC posteriors: FLAMINGO B=1 counts, $Y_{5R500c}$ vs $Y_{R500c}$ (dashed = truth)",
               fontsize=12, y=1.02)
g.export(os.path.join(OUTDIR,"cnc_triangle_all.pdf")); g.export(os.path.join(OUTDIR,"cnc_triangle_all.png"), dpi=300)
plt.show()


## Omega_m - sigma8 plane (counts S8 degeneracy)

In [3]:
from matplotlib.lines import Line2D
g2=plots.get_single_plotter(width_inch=5.5)
chains = [s] if s1_Y500c is None else [s, s1_Y500c]
colors = ["#1f77b4"] if s1_Y500c is None else ["#1f77b4", "#2ca02c"]
g2.plot_2d(chains, "Omega_m", "sigma8", filled=True, colors=colors)
ax=g2.get_axes()
ax.plot(OM_TRUTH, S8_TRUTH, "*", color="crimson", ms=16, mec="k", zorder=5)
om=np.linspace(*ax.get_xlim(),50); S8t=S8_TRUTH*np.sqrt(OM_TRUTH/0.3)
ax.plot(om, S8t/np.sqrt(om/0.3), "k--", lw=1, alpha=0.6)
handles=[
    Line2D([],[],color="#1f77b4",lw=8,alpha=0.7,label=f"B=1 Y_5R500c (N={N_Y5R500C})"),
]
if s1_Y500c is not None:
    handles.append(Line2D([],[],color="#2ca02c",lw=8,alpha=0.7,label=f"B=1 Y_R500c (N={N_Y500C})"))
handles += [
    Line2D([],[],color="crimson",marker="*",ls="",ms=12,mec="k",label="FLAMINGO truth"),
    Line2D([],[],color="k",ls="--",lw=1,alpha=0.6,label=r"$S_8$ = truth"),
]
ax.set_xlabel(r"$\Omega_m$"); ax.set_ylabel(r"$\sigma_8$")
ax.legend(handles=handles, fontsize=9)
ax.set_title(r"B=1: $Y_{5R500c}$ vs $Y_{R500c}$", fontsize=11)
g2.export(os.path.join(OUTDIR,"cnc_Om_s8_plane.pdf")); g2.export(os.path.join(OUTDIR,"cnc_Om_s8_plane.png"), dpi=300)
print("saved ->",OUTDIR); plt.show()


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb27_cnc_cosmo_contours


## Notes

- Five parameters sampled; A_SZ and sigma_lnY are nuisances; B fixed per selection.
- A separate B=1.35 chain using **SOAP Y_R500c** (GNFW K at r_out=1) is in the
  section below (`cnc_cosmo_arnaudB135_Y500c`, N(q>5)=768 vs 885 for Y_5R500c).

## B=1.35 selection (sigma8 / omega_cdm priors)

The B=1.35 cluster selection recomputes SNR under hydrostatic bias B (cosmocnc
convention 1-b = 1/B): y0 ~ (M/B)^alpha and theta_500 ~ (M/B)^(1/3), giving
N(q>5)=885 (vs 2392 at B=1). Chain below uses the **original** priors
(sample `omega_cdm`, `sigma8`), identical to the B=1 run apart from data and B=1.35.

In [4]:
CHAIN_B135=os.path.join(REPO,"chains","cnc_cosmo_arnaudB135","cnc_cosmo")
s135=loadMCSamples(CHAIN_B135, settings={"ignore_rows":0.3})
p135=s135.getParams()
s135.addDerived(p135.sigma8*np.sqrt(p135.Omega_m/0.3), name="S8", label=r"S_8\equiv\sigma_8(\Omega_m/0.3)^{0.5}")
print("CNC posterior B=1.35 (mean +/- std):")
for nm in PARS:
    print(f"  {nm:10s} = {s135.mean(nm):.4f} +/- {s135.std(nm):.4f}")
print(f"\nFLAMINGO truth: Omega_m={OM_TRUTH}, sigma8={S8_TRUTH}, S8={S8_TRUTH*np.sqrt(OM_TRUTH/0.3):.3f}")
print("B=1   S8 = %.4f +/- %.4f" % (s.mean("S8"), s.std("S8")))
print("B=1.35 S8 = %.4f +/- %.4f" % (s135.mean("S8"), s135.std("S8")))

CNC posterior B=1.35 (mean +/- std):
  Omega_m    = 0.3649 +/- 0.0270
  sigma8     = 0.7195 +/- 0.0149
  S8         = 0.7927 +/- 0.0276
  h          = 0.6833 +/- 0.0237
  omega_b    = 0.0225 +/- 0.0021
  n_s        = 0.9676 +/- 0.0141
  alpha_SZ   = 1.0991 +/- 0.0643
  A_SZ       = -4.3221 +/- 0.0568
  sigma_lnY  = 0.1737 +/- 0.0219

FLAMINGO truth: Omega_m=0.306, sigma8=0.808, S8=0.816
B=1   S8 = 0.7904 +/- 0.0247
B=1.35 S8 = 0.7927 +/- 0.0276


### Triangle: B=1 vs B=1.35 (sigma8 priors)

In [5]:
g3=plots.get_subplot_plotter(width_inch=11.0)
g3.settings.alpha_filled_add=0.7; g3.settings.title_limit_fontsize=10
g3.triangle_plot([s, s135], PARS, filled=True, contour_colors=["#1f77b4","#d62728"],
                 legend_labels=["B=1 (N=2392)","B=1.35 sigma8 (N=885)"], legend_loc="upper right",
                 title_limit=1, markers=TRUTH)
g3.fig.suptitle(r"CNC posteriors: FLAMINGO counts, $B=1$ vs $B=1.35$ (dashed = truth/fiducial)", fontsize=12, y=1.02)
g3.export(os.path.join(OUTDIR,"cnc_triangle_B1_vs_B135.pdf")); g3.export(os.path.join(OUTDIR,"cnc_triangle_B1_vs_B135.png"), dpi=300)
plt.show()


### Omega_m - sigma8 plane: B=1 vs B=1.35

In [6]:
from matplotlib.lines import Line2D
g4=plots.get_single_plotter(width_inch=5.5)
g4.plot_2d([s, s135], "Omega_m", "sigma8", filled=True, colors=["#1f77b4","#d62728"])
ax=g4.get_axes()
ax.plot(OM_TRUTH, S8_TRUTH, "*", color="crimson", ms=16, mec="k", zorder=5)
om=np.linspace(*ax.get_xlim(),50); S8t=S8_TRUTH*np.sqrt(OM_TRUTH/0.3)
ax.plot(om, S8t/np.sqrt(om/0.3), "k--", lw=1, alpha=0.6)
handles=[Line2D([],[],color="#1f77b4",lw=8,alpha=0.7,label="B=1"),
         Line2D([],[],color="#d62728",lw=8,alpha=0.7,label="B=1.35 (sigma8)"),
         Line2D([],[],color="crimson",marker="*",ls="",ms=12,mec="k",label="FLAMINGO truth"),
         Line2D([],[],color="k",ls="--",lw=1,alpha=0.6,label=r"$S_8$ = truth")]
ax.set_xlabel(r"$\Omega_m$"); ax.set_ylabel(r"$\sigma_8$")
ax.legend(handles=handles, fontsize=9); ax.set_title("CNC counts degeneracy: $B=1$ vs $B=1.35$",fontsize=11)
g4.export(os.path.join(OUTDIR,"cnc_Om_s8_plane_B1_vs_B135.pdf")); g4.export(os.path.join(OUTDIR,"cnc_Om_s8_plane_B1_vs_B135.png"), dpi=300)
print("saved ->",OUTDIR); plt.show()


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb27_cnc_cosmo_contours


## B=1.35 with SOAP $Y_{R500c}$ (GNFW $r_\mathrm{out}=1$)

Same B=1.35 priors as above, but SNR built from SOAP `Y_500c_Mpc2` with the GNFW
geometric factor at $r_\mathrm{out}=1$ (not $Y_{5R500c}$ at $r_\mathrm{out}=5$).
Gives N(q>5)=768 (vs 885 for $Y_{5R500c}$). Config:
`cobaya/configs/cnc_cosmo_arnaudB135_Y500c.yaml`.

In [7]:
N_Y500C = 768   # q>5 from Y_R500c catalogue (vs 885 for Y_5R500c)

CHAIN_B135_Y500C = os.path.join(REPO, "chains", "cnc_cosmo_arnaudB135_Y500c", "cnc_cosmo")
if not os.path.exists(CHAIN_B135_Y500C + ".1.txt"):
    print("chain not found:", CHAIN_B135_Y500C)
    print("run: bash cobaya/run_cnc_cosmo_B135_Y500c.sh")
    s135_Y500c = None
else:
    s135_Y500c = loadMCSamples(CHAIN_B135_Y500C, settings={"ignore_rows": 0.3})
    p = s135_Y500c.getParams()
    s135_Y500c.addDerived(p.sigma8 * np.sqrt(p.Omega_m / 0.3), name="S8",
                          label=r"S_8\equiv\sigma_8(\Omega_m/0.3)^{0.5}")
    print(f"CNC posterior B=1.35 Y_R500c (N={N_Y500C}; same priors as Y_5R500c chain):")
    for nm in PARS:
        if nm not in s135_Y500c.getParamNames().list():
            continue
        print(f"  {nm:10s} = {s135_Y500c.mean(nm):.4f} +/- {s135_Y500c.std(nm):.4f}")
    print(f"\nFLAMINGO truth: Omega_m={OM_TRUTH}, sigma8={S8_TRUTH}, S8={S8_TRUTH*np.sqrt(OM_TRUTH/0.3):.3f}")
    print("B=1.35 Y_5R500c S8 = %.4f +/- %.4f" % (s135.mean("S8"), s135.std("S8")))
    print("B=1.35 Y_R500c  S8 = %.4f +/- %.4f" % (s135_Y500c.mean("S8"), s135_Y500c.std("S8")))

CNC posterior B=1.35 Y_R500c (N=768; same priors as Y_5R500c chain):
  Omega_m    = 0.2942 +/- 0.0241
  sigma8     = 0.7629 +/- 0.0259
  S8         = 0.7542 +/- 0.0267
  h          = 0.6800 +/- 0.0236
  omega_b    = 0.0225 +/- 0.0020
  n_s        = 0.9682 +/- 0.0141
  alpha_SZ   = 1.0639 +/- 0.0748
  A_SZ       = -4.2954 +/- 0.0581
  sigma_lnY  = 0.1750 +/- 0.0228

FLAMINGO truth: Omega_m=0.306, sigma8=0.808, S8=0.816
B=1.35 Y_5R500c S8 = 0.7927 +/- 0.0276
B=1.35 Y_R500c  S8 = 0.7542 +/- 0.0267


### Triangle: B=1.35 $Y_{5R500c}$ vs $Y_{R500c}$

In [8]:
if s135_Y500c is not None:
    g5 = plots.get_subplot_plotter(width_inch=11.0)
    g5.settings.alpha_filled_add = 0.7; g5.settings.title_limit_fontsize = 10
    g5.triangle_plot([s135, s135_Y500c], PARS, filled=True,
                     contour_colors=["#d62728", "#2ca02c"],
                     legend_labels=["B=1.35 Y_5R500c (N=885)", f"B=1.35 Y_R500c (N={N_Y500C})"],
                     legend_loc="upper right", title_limit=1, markers=TRUTH)
    g5.fig.suptitle(r"CNC B=1.35: $Y_{5R500c}$ vs $Y_{R500c}$ (dashed = truth)", fontsize=12, y=1.02)
    g5.export(os.path.join(OUTDIR, "cnc_triangle_B135_Y500c.pdf"))
    g5.export(os.path.join(OUTDIR, "cnc_triangle_B135_Y500c.png"), dpi=300)
    plt.show()


### $\Omega_m$ - $\sigma_8$ plane: $Y_{5R500c}$ vs $Y_{R500c}$

In [9]:
if s135_Y500c is not None:
    from matplotlib.lines import Line2D
    g6 = plots.get_single_plotter(width_inch=5.5)
    g6.plot_2d([s135, s135_Y500c], "Omega_m", "sigma8", filled=True,
               colors=["#d62728", "#2ca02c"])
    ax = g6.get_axes()
    ax.plot(OM_TRUTH, S8_TRUTH, "*", color="crimson", ms=16, mec="k", zorder=5)
    om = np.linspace(*ax.get_xlim(), 50); S8t = S8_TRUTH * np.sqrt(OM_TRUTH / 0.3)
    ax.plot(om, S8t / np.sqrt(om / 0.3), "k--", lw=1, alpha=0.6)
    handles = [
        Line2D([], [], color="#d62728", lw=8, alpha=0.7, label="B=1.35 Y_5R500c (N=885)"),
        Line2D([], [], color="#2ca02c", lw=8, alpha=0.7, label=f"B=1.35 Y_R500c (N={N_Y500C})"),
        Line2D([], [], color="crimson", marker="*", ls="", ms=12, mec="k", label="FLAMINGO truth"),
        Line2D([], [], color="k", ls="--", lw=1, alpha=0.6, label=r"$S_8$ = truth"),
    ]
    ax.set_xlabel(r"$\Omega_m$"); ax.set_ylabel(r"$\sigma_8$")
    ax.legend(handles=handles, fontsize=9)
    ax.set_title(r"B=1.35: $Y_{5R500c}$ vs $Y_{R500c}$", fontsize=11)
    g6.export(os.path.join(OUTDIR, "cnc_Om_s8_B135_Y500c.pdf"))
    g6.export(os.path.join(OUTDIR, "cnc_Om_s8_B135_Y500c.png"), dpi=300)
    print("saved ->", OUTDIR); plt.show()


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb27_cnc_cosmo_contours
